# BioSCape AVIRIS-NG QC: CRS, Transforms, and Processing Metadata

In [1]:
from pathlib import Path
import importlib, sys, warnings
import numpy as np
import pandas as pd
import netCDF4 as nc
import rasterio
from pyproj import CRS, Transformer
warnings.filterwarnings("ignore")

REPO = Path("Y:/personal/lberberian/BioSCape/BioSCape-AVRISNG_Spectra")
RFL_SUB = REPO / "data" / "rfl_ocean_subset"
SCENES = REPO / "outputs" / "scenes"

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import bioscape_rfl_tools as brt
importlib.reload(brt)

nc_paths = sorted(RFL_SUB.glob("*.nc"))
print(f"scenes: {len(nc_paths)}")

scenes: 50


## 1. CRS and GeoTransform: .nc vs TIF

In [2]:
rows = []
for nc_p in nc_paths:
    sid = nc_p.stem
    r = {"scene_id": sid}
    with nc.Dataset(nc_p, "r") as ds:
        gm = ds.variables["transverse_mercator"]
        nc_wkt = gm.getncattr("spatial_ref") if "spatial_ref" in gm.ncattrs() else None
        r["nc_epsg"] = CRS.from_wkt(nc_wkt).to_epsg() if nc_wkt else None
        gt = [float(v) for v in gm.getncattr("GeoTransform").split()]
        r["nc_gt_x0"], r["nc_gt_dx"] = gt[0], gt[1]
        r["nc_gt_y0"], r["nc_gt_dy"] = gt[3], gt[5]
        spec = brt.load_spec(ds)
        r["brt_epsg"] = spec.epsg
    for tag, suffix in [("rgb", "_rgb_u8.tif"), ("ndvi", "_ndvi.tif")]:
        tif = SCENES / sid / "quicklooks" / f"{sid}{suffix}"
        if tif.exists():
            with rasterio.open(tif) as src:
                r[f"{tag}_epsg"] = src.crs.to_epsg()
                r[f"{tag}_x0"] = src.transform.c
                r[f"{tag}_y0"] = src.transform.f
                r[f"{tag}_dx"] = src.transform.a
                r[f"{tag}_dy"] = src.transform.e
        else:
            r[f"{tag}_epsg"] = None
    r["crs_match"] = (r["nc_epsg"] == r["brt_epsg"] == r.get("rgb_epsg") == r.get("ndvi_epsg"))
    r["gt_match"] = all([
        r.get("rgb_x0") is not None,
        abs(r["nc_gt_x0"] - r.get("rgb_x0", 0)) < 0.01,
        abs(r["nc_gt_y0"] - r.get("rgb_y0", 0)) < 0.01,
    ])
    rows.append(r)

crs_df = pd.DataFrame(rows)
print(f"CRS match across .nc / brt / rgb / ndvi : {crs_df['crs_match'].sum()} / {len(crs_df)}")
print(f"GeoTransform origin match .nc vs TIF    : {crs_df['gt_match'].sum()} / {len(crs_df)}")
fails = crs_df[~crs_df["crs_match"] | ~crs_df["gt_match"]]
if len(fails):
    print("\nFailed:")
    print(fails[["scene_id","nc_epsg","brt_epsg","rgb_epsg","ndvi_epsg","crs_match","gt_match"]].to_string(index=False))
else:
    print("All scenes pass.")

CRS match across .nc / brt / rgb / ndvi : 50 / 50
GeoTransform origin match .nc vs TIF    : 50 / 50
All scenes pass.


## 2. Pixel values: .nc reflectance to 8-bit RGB

In [3]:
RGB_WLS = (650.0, 560.0, 470.0)
STRETCH_P = (1.0, 99.0)
VALID_RNG = (0.0, 1.5)
pix_rows = []

for i, nc_p in enumerate(nc_paths):
    sid = nc_p.stem
    rgb_tif = SCENES / sid / "quicklooks" / f"{sid}_rgb_u8.tif"
    r = {"scene_id": sid}
    if not rgb_tif.exists():
        r["error"] = "missing"
        pix_rows.append(r)
        continue
    try:
        with nc.Dataset(nc_p, "r") as ds:
            spec = brt.load_spec(ds)
            nc_bands = []
            for wl in RGB_WLS:
                bi = brt.closest_band(spec.wl, wl)
                band = brt._as_float32(brt.read_band(ds, spec, bi))
                _, _, band = brt._fix_xy_orientation(spec.x, spec.y, band)
                band = brt._mask_invalid(band, *VALID_RNG)
                band = brt._stretch_percentile(band, *STRETCH_P, vmin=VALID_RNG[0], vmax=VALID_RNG[1])
                nc_bands.append(brt._scale_to_u8(band))
        nc_rgb = np.stack(nc_bands, axis=0)
        with rasterio.open(rgb_tif) as src:
            tif_rgb = src.read()
        diff = np.abs(nc_rgb.astype(int) - tif_rgb.astype(int))
        r["max_diff"] = int(diff.max())
        r["mean_diff"] = float(diff.mean())
        r["error"] = None
    except Exception as e:
        r["error"] = str(e)
    pix_rows.append(r)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(nc_paths)}")

pix_df = pd.DataFrame(pix_rows)
ok = pix_df[pix_df["error"].isna()]
print(f"\nRGB pixel match (.nc -> stretch -> u8 vs TIF)")
print(f"Compared : {len(ok)}  Max diff : {ok['max_diff'].max()}  Mean diff : {ok['mean_diff'].mean():.4f}")
bad = ok[ok["max_diff"] > 0]
if len(bad):
    print(f"{len(bad)} scene(s) with differences:")
    print(bad[["scene_id","max_diff","mean_diff"]].to_string(index=False))
else:
    print("All scenes match exactly.")

  10/50
  20/50
  30/50
  40/50
  50/50

RGB pixel match (.nc -> stretch -> u8 vs TIF)
Compared : 50  Max diff : 0  Mean diff : 0.0000
All scenes match exactly.


## 3. NDVI values: .nc bands vs NDVI TIF

In [4]:
NDVI_WLS = (670.0, 750.0)
ndvi_rows = []

for i, nc_p in enumerate(nc_paths):
    sid = nc_p.stem
    ndvi_tif = SCENES / sid / "quicklooks" / f"{sid}_ndvi.tif"
    r = {"scene_id": sid}
    if not ndvi_tif.exists():
        r["error"] = "missing"
        ndvi_rows.append(r)
        continue
    try:
        with nc.Dataset(nc_p, "r") as ds:
            spec = brt.load_spec(ds)
            red = brt._as_float32(brt.read_band(ds, spec, brt.closest_band(spec.wl, NDVI_WLS[0])))
            nir = brt._as_float32(brt.read_band(ds, spec, brt.closest_band(spec.wl, NDVI_WLS[1])))
            _, _, red = brt._fix_xy_orientation(spec.x, spec.y, red)
            _, _, nir = brt._fix_xy_orientation(spec.x, spec.y, nir)
            red = brt._mask_invalid(red, *VALID_RNG)
            nir = brt._mask_invalid(nir, *VALID_RNG)
            ndvi_nc = (nir - red) / (nir + red + 1e-12)
            ndvi_nc = brt._mask_invalid(ndvi_nc, -1.0, 1.0)
        with rasterio.open(ndvi_tif) as src:
            ndvi_disk = src.read(1).astype("float32")
            if src.nodata is not None:
                ndvi_disk[ndvi_disk == src.nodata] = np.nan
        both = np.isfinite(ndvi_nc) & np.isfinite(ndvi_disk)
        if both.sum() > 0:
            diff = np.abs(ndvi_nc[both] - ndvi_disk[both])
            r["max_diff"] = float(diff.max())
            r["mean_diff"] = float(diff.mean())
        r["error"] = None
    except Exception as e:
        r["error"] = str(e)
    ndvi_rows.append(r)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(nc_paths)}")

ndvi_df = pd.DataFrame(ndvi_rows)
ok_n = ndvi_df[ndvi_df["error"].isna()]
print(f"\nNDVI match (.nc bands vs TIF)")
print(f"Compared : {len(ok_n)}  Max diff : {ok_n['max_diff'].max():.8f}")
bad_n = ok_n[ok_n["max_diff"] > 0.001]
if len(bad_n):
    print(f"{len(bad_n)} scene(s) with differences > 0.001:")
    print(bad_n[["scene_id","max_diff"]].to_string(index=False))
else:
    print("All scenes match exactly.")

  10/50
  20/50
  30/50
  40/50
  50/50

NDVI match (.nc bands vs TIF)
Compared : 50  Max diff : 0.00000000
All scenes match exactly.


## 4. 8-bit stretch quality

In [5]:
stretch_rows = []
for nc_p in nc_paths:
    sid = nc_p.stem
    rgb_tif = SCENES / sid / "quicklooks" / f"{sid}_rgb_u8.tif"
    r = {"scene_id": sid}
    if not rgb_tif.exists():
        stretch_rows.append(r)
        continue
    with rasterio.open(rgb_tif) as src:
        rgb = src.read()
    nodata = (rgb[0]==0)&(rgb[1]==0)&(rgb[2]==0)
    valid = ~nodata
    n_valid = int(valid.sum())
    r["pct_nodata"] = round(nodata.sum() / nodata.size * 100, 1)
    if n_valid > 0:
        vals = rgb[:, valid].astype(float)
        r["mean_brightness"] = round(float(vals.mean()), 1)
        r["max_clip_255_pct"] = round(max((rgb[b][valid]==255).mean()*100 for b in range(3)), 1)
        flags = []
        if r["mean_brightness"] < 30: flags.append("very_dark")
        if r["mean_brightness"] > 200: flags.append("very_bright")
        if r["pct_nodata"] > 80: flags.append("mostly_nodata")
        if r["max_clip_255_pct"] > 10: flags.append("heavy_clipping")
        r["flags"] = ", ".join(flags) if flags else None
    stretch_rows.append(r)

stretch_df = pd.DataFrame(stretch_rows)
flagged = stretch_df[stretch_df["flags"].notna()]
print(f"8-bit stretch: {len(stretch_df)} scenes checked")
print(f"Brightness range: {stretch_df['mean_brightness'].min():.0f} - {stretch_df['mean_brightness'].max():.0f} / 255")
if len(flagged):
    print(f"\n{len(flagged)} scene(s) flagged:")
    print(flagged[["scene_id","mean_brightness","pct_nodata","flags"]].to_string(index=False))
else:
    print("No stretch issues.")

8-bit stretch: 50 scenes checked
Brightness range: 19 - 119 / 255

11 scene(s) flagged:
                                      scene_id  mean_brightness  pct_nodata     flags
ang20231029t104631_000_L2A_OE_904bad5c_RFL_ORT             27.6        32.1 very_dark
ang20231031t072606_002_L2A_OE_0b4f48b4_RFL_ORT             26.3        60.9 very_dark
ang20231109t071216_002_L2A_OE_0b4f48b4_RFL_ORT             26.0        37.2 very_dark
ang20231109t071216_008_L2A_OE_0b4f48b4_RFL_ORT             28.0        40.6 very_dark
ang20231109t073859_001_L2A_OE_0b4f48b4_RFL_ORT             19.0        38.2 very_dark
ang20231109t073859_003_L2A_OE_0b4f48b4_RFL_ORT             27.6        39.1 very_dark
ang20231109t073859_005_L2A_OE_0b4f48b4_RFL_ORT             26.0        39.2 very_dark
ang20231109t080214_008_L2A_OE_0b4f48b4_RFL_ORT             28.6        46.1 very_dark
ang20231109t135449_001_L2A_OE_0b4f48b4_RFL_ORT             22.8        15.8 very_dark
ang20231110t064703_008_L2A_OE_0b4f48b4_RFL_ORT      

## 5. Global attributes: all scenes

In [6]:
meta_rows = []
all_attr_keys = set()

for nc_p in nc_paths:
    sid = nc_p.stem
    with nc.Dataset(nc_p, "r") as ds:
        r = {"scene_id": sid}
        for attr in ds.ncattrs():
            val = ds.getncattr(attr)
            if isinstance(val, str) and len(val) > 120:
                val = val[:120] + "..."
            r[attr] = val
            all_attr_keys.add(attr)
        meta_rows.append(r)

meta_df = pd.DataFrame(meta_rows)

In [7]:
key_attrs = ["software_build_version", "product_version", "platform",
             "date_created", "flight_line", "scene"]

print("Attribute value counts across all scenes")
print("=" * 45)
for attr in key_attrs:
    if attr in meta_df.columns:
        print(f"\n{attr}:")
        print(meta_df[attr].value_counts().to_string())

Attribute value counts across all scenes

software_build_version:
software_build_version
002              46
010200_rdndev     4

product_version:
product_version
1       46
test     4

platform:
platform
Gulfstream III    46
Not specified      4

date_created:
date_created
2024-11-25T22:44:47Z    1
2024-11-25T21:49:51Z    1
2024-11-26T01:19:30Z    1
2024-12-02T10:45:13Z    1
2024-11-25T20:34:21Z    1
2024-11-25T20:30:15Z    1
2024-11-26T02:34:43Z    1
2024-11-26T02:46:05Z    1
2025-02-26T15:37:28Z    1
2025-02-26T15:37:44Z    1
2025-02-26T15:37:46Z    1
2025-02-26T15:37:45Z    1
2024-11-26T10:13:53Z    1
2024-11-26T10:55:18Z    1
2024-11-26T10:55:04Z    1
2024-11-26T11:20:31Z    1
2024-11-26T21:00:50Z    1
2024-11-27T01:37:13Z    1
2024-11-26T22:27:35Z    1
2024-11-26T22:27:36Z    1
2024-11-26T22:26:17Z    1
2024-11-26T22:51:19Z    1
2024-11-27T01:35:39Z    1
2024-11-27T01:35:36Z    1
2024-11-27T01:35:37Z    1
2024-11-27T01:35:58Z    1
2024-11-27T01:36:00Z    1
2024-11-26T21:28:28Z   

In [8]:
non_prod = meta_df[
    (meta_df.get("product_version",pd.Series(dtype=str)) == "test") |
    (meta_df.get("software_build_version",pd.Series(dtype=str)).str.contains("dev", case=False, na=False)) |
    (meta_df.get("platform",pd.Series(dtype=str)) == "Not specified")
]

print(f"{len(non_prod)} scene(s) with non-production metadata:")
if len(non_prod):
    print(non_prod[["scene_id"] + [a for a in key_attrs if a in non_prod.columns]].to_string(index=False))

4 scene(s) with non-production metadata:
                                      scene_id software_build_version product_version      platform         date_created        flight_line scene
ang20231029t104631_000_L2A_OE_904bad5c_RFL_ORT          010200_rdndev            test Not specified 2025-02-26T15:37:28Z ang20231029t104631   000
ang20231029t104631_001_L2A_OE_904bad5c_RFL_ORT          010200_rdndev            test Not specified 2025-02-26T15:37:44Z ang20231029t104631   001
ang20231029t104631_002_L2A_OE_904bad5c_RFL_ORT          010200_rdndev            test Not specified 2025-02-26T15:37:46Z ang20231029t104631   002
ang20231029t104631_003_L2A_OE_904bad5c_RFL_ORT          010200_rdndev            test Not specified 2025-02-26T15:37:45Z ang20231029t104631   003


In [9]:
print("Full global attributes for the 4 non-production scenes")
print("=" * 55)
for _, row in non_prod.iterrows():
    print(f"\n{row['scene_id']}")
    for col in sorted(all_attr_keys):
        if col in row and col != "scene_id":
            print(f"  {col}: {row[col]}")

Full global attributes for the 4 non-production scenes

ang20231029t104631_000_L2A_OE_904bad5c_RFL_ORT
  Conventions: CF-1.6
  creator_name: Jet Propulsion Laboratory/California Institute of Technology
  creator_url: aviris.jpl.nasa.gov
  date_created: 2025-02-26T15:37:28Z
  flight_line: ang20231029t104631
  identifier_product_doi_authority: 
  institution: NASA Jet Propulsion Laboratory/California Institute of Technology
  instrument: AVIRIS-NG
  keywords: Imaging Spectroscopy, AVIRIS, AVIRIS-NG
  keywords_vocabulary: NASA Global Change Master Directory (GCMD) Science Keywords
  ncei_template_version: NCEI_NetCDF_Grid_Template_v2.0
  platform: Not specified
  processing_level: L2A
  product_version: test
  publisher_institution: 
  publisher_type: 
  scene: 000
  sensor: Airborne Visible / Infrared Imaging Spectrometer Next Generation
  software_build_version: 010200_rdndev
  summary: The Airborne Visible / Infrared Imaging Spectrometer - Next Generation (AVIRIS-NG) is part of NASA’s 

In [10]:
print("Attributes that differ between production and non-production scenes")
print("=" * 65)
prod = meta_df[~meta_df.index.isin(non_prod.index)]

for attr in sorted(all_attr_keys):
    if attr not in meta_df.columns or attr == "scene_id":
        continue
    prod_vals = set(prod[attr].dropna().unique())
    test_vals = set(non_prod[attr].dropna().unique())
    if prod_vals != test_vals:
        print(f"\n{attr}:")
        print(f"  production ({len(prod)} scenes) : {prod_vals}")
        print(f"  test       ({len(non_prod)} scenes) : {test_vals}")

Attributes that differ between production and non-production scenes

date_created:
  production (46 scenes) : {'2024-11-26T01:19:30Z', '2024-11-25T20:30:15Z', '2024-11-27T10:56:21Z', '2024-11-25T21:49:51Z', '2024-11-26T02:34:43Z', '2024-11-27T14:44:14Z', '2024-11-27T18:21:05Z', '2024-11-26T22:51:19Z', '2024-11-26T21:17:56Z', '2024-11-27T07:59:58Z', '2024-11-27T07:57:55Z', '2024-11-27T12:52:33Z', '2024-11-27T15:54:38Z', '2024-11-27T05:50:18Z', '2024-11-27T17:54:05Z', '2024-11-27T04:01:27Z', '2024-11-27T07:59:30Z', '2024-11-27T06:26:12Z', '2024-11-26T22:27:35Z', '2024-11-27T01:37:13Z', '2024-11-26T11:20:31Z', '2024-11-26T22:27:36Z', '2024-11-26T21:28:28Z', '2024-11-27T01:35:39Z', '2024-11-26T21:00:50Z', '2024-11-26T22:26:17Z', '2024-11-27T05:37:56Z', '2024-11-25T22:44:47Z', '2024-11-27T08:09:07Z', '2024-11-27T06:42:38Z', '2024-12-02T10:45:13Z', '2024-11-28T12:55:29Z', '2024-11-27T04:16:29Z', '2024-11-27T05:29:01Z', '2024-11-27T16:44:20Z', '2024-11-27T01:35:58Z', '2024-11-27T01:35:37Z', '